# 04 特徴量重要度 — 「何が効いているか」を、どう調べるか

## このノートの目的
- **impurity による重要度**と **permutation による重要度**の違いを、人工データで確かめる。
- 実データ（那覇）で、どの特徴量・どの**群**が効いているかを見る。
- 重要度を読むときの**落とし穴**（似た特徴量・偏り）を知る。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **impurity（不純度）による重要度** | 木が分岐するときに**不純度を減らした量**を、特徴量ごとに合計したもの。学習データだけで計算でき、速い。 |
| **permutation（並べ替え）による重要度** | 評価データで、ある特徴量の値を**ランダムに並べ替え**（その特徴量の情報を壊し）、**PR-AUC がどれだけ下がるか**を見る。下がるほど重要。 |
| **群（グループ）** | 似た意味の特徴量をまとめたもの（例: 露点・湿度・露点差 ＝「水蒸気」）。まとめて並べ替える。 |
| **相関のある特徴量** | 互いに似た値を取る特徴量。片方を壊しても、もう一方が代わりをするので、**重要度が低く見える**。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score

## 1. 人工データで、impurity の「偏り」を見る

答えが分かっている人工データを作ります。**正解に関係があるのは `signal` だけ**で、ほかの2つは**ただのノイズ**です。

| 特徴量 | 中身 |
|---|---|
| `signal` | 0か1。**1のとき正解になりやすい**（本当に効く特徴量） |
| `noise_cont` | 連続値のランダムな数（**関係ない**。値の種類が非常に多い） |
| `noise_bin` | 0か1のランダム（**関係ない**。値の種類が少ない） |

In [ ]:
# 目的: 答えの分かっている人工データで、impurity と permutation の重要度を比べる
rng = np.random.default_rng(0)
n = 4000
d = pd.DataFrame({"signal": rng.integers(0, 2, n), "noise_cont": rng.normal(size=n), "noise_bin": rng.integers(0, 2, n)})
yv = (rng.random(n) < 0.05 + 0.20 * d["signal"]).astype(int)           # signal が1なら25%、0なら5%で陽性
tr, te = np.arange(n) < n // 2, np.arange(n) >= n // 2
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=1, random_state=0, n_jobs=-1).fit(d[tr], yv[tr])

imp = pd.Series(rf.feature_importances_, index=d.columns)                              # impurity（学習データで計算）
pm = permutation_importance(rf, d[te], yv[te], scoring="average_precision", n_repeats=20, random_state=0)
perm = pd.Series(pm.importances_mean, index=d.columns)                                 # permutation（評価データで計算）
res = pd.DataFrame({"impurity": imp, "permutation（PR-AUCの低下量）": perm}).round(3)
res

**出力の見方**
- **impurity**: 本当は無関係な `noise_cont`（連続値）が、**高い重要度**になりがちです。値の種類が多いと、偶然うまく分けられるしきい値が見つかりやすく、**不純度が下がったように見える**ためです。
- **permutation**: `noise_cont` は**ほぼ0**（並べ替えても、評価データの PR-AUC は下がらない）で、**`signal` だけが大きく**出ます。
- つまり、impurity は**連続値・値の種類が多い特徴量を過大評価**しやすく、permutation は**評価データでの本当の効き**を測ります。

> 📝 impurity は学習データで計算するので、**過学習した分も重要度に含めてしまう**のが原因です。

## 2. 実データ（那覇）で見る

2010〜2019年を学習、2020〜2024年を評価にした簡易版で、ExtraTrees の重要度を比べます（ノート02と同じ分け方）。

In [ ]:
# 目的: 実データの用意と、ExtraTrees の学習
x = ds.build("naha"); f = ds.features("naha", change=False); target = "y20_region"
dev = x[(x.year <= 2024) & x.wind_ok].reset_index(drop=True)
trn, val = dev[dev.year <= 2019], dev[dev.year >= 2020]
med = trn[f].median()
Xtr, ytr, Xva, yva = trn[f].fillna(med), trn[target], val[f].fillna(med), val[target]
et = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=0).fit(Xtr, ytr)
print(f"評価側: {len(val)}標本、陽性 {yva.sum()}、PR-AUC = {average_precision_score(yva, et.predict_proba(Xva)[:, 1]):.3f}")

imp = pd.Series(et.feature_importances_, index=f)
pm = permutation_importance(et, Xva, yva, scoring="average_precision", n_repeats=10, random_state=0, n_jobs=-1)
tab = pd.DataFrame({"impurity": imp, "permutation": pm.importances_mean, "permutation のばらつき": pm.importances_std})
tab["impurity の順位"] = tab["impurity"].rank(ascending=False).astype(int)
tab["permutation の順位"] = tab["permutation"].rank(ascending=False).astype(int)
tab.sort_values("permutation", ascending=False).round(4)

In [ ]:
# 目的: 2つの重要度を並べて棒グラフにする
order = tab.sort_values("permutation").index
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
ax[0].barh(order, tab.loc[order, "impurity"], color="#4c72b0"); ax[0].set_title("impurity（学習データで計算）")
ax[1].barh(order, tab.loc[order, "permutation"], xerr=tab.loc[order, "permutation のばらつき"], color="#c44e52"); ax[1].set_title("permutation（評価データで計算）\nPR-AUC の低下量")
ax[1].axvline(0, color="k", lw=0.5); plt.tight_layout(); plt.show()
from scipy.stats import spearmanr
print("順位の相関（スピアマン）:", round(spearmanr(tab["impurity"], tab["permutation"]).statistic, 2), "（1に近いほど、順位が似ている）")

**出力の見方**
- 左（impurity）と右（permutation）で、**順位がかなり違う**特徴量があります。たとえば、**月や風向**のように、値の種類が多いか、季節で偏る特徴量は、impurity で上位に出やすいです。
- permutation の値が**マイナス**の特徴量は、「並べ替えたほうが少し良くなった」＝**効いていない**（むしろノイズ）という意味です。棒の横線（ばらつき）が0をまたぐなら、**効いているとは言えません**。
- 順位相関が 1 から遠いほど、**impurity だけで判断するのは危険**です。

## 3. 群で並べ替える

露点・湿度・露点差は、**互いに似た情報**を持っています。1つずつ並べ替えると、**残りの特徴量が代わりを務めて**、重要度が低く見えます。
そこで、**似たものをまとめて**並べ替えます。

In [ ]:
# 目的: 特徴量を意味ごとの「群」にまとめ、群ごとにまとめて並べ替えて、PR-AUC の低下量を見る
groups = {"気圧": ["pres", "pres_d1", "pres_d3"], "気温": ["temp"], "水蒸気（露点・湿度）": ["dew", "dewdep", "humid"],
          "風": ["wind", "wdir_sin", "wdir_cos", "wind_mean", "wind_max"], "直近の降水": ["rain1", "rain3", "rain6", "rainmax6"],
          "午前の日照・日射": ["sun_morning", "rad_morning"], "時刻・月": ["t", "month"]}
base = average_precision_score(yva, et.predict_proba(Xva)[:, 1])
rng = np.random.default_rng(0)
rows = []
for g, cols in groups.items():
    cols = [c for c in cols if c in f]
    drops = []
    for _ in range(10):
        Xp = Xva.copy()
        perm_idx = rng.permutation(len(Xva))
        Xp[cols] = Xva[cols].iloc[perm_idx].to_numpy()                   # 群の列を、同じ並べ替えで一緒に入れ替える
        drops.append(base - average_precision_score(yva, et.predict_proba(Xp)[:, 1]))
    rows.append({"群": g, "PR-AUC の低下量（平均）": np.mean(drops), "ばらつき": np.std(drops)})
gt = pd.DataFrame(rows).set_index("群").sort_values("PR-AUC の低下量（平均）", ascending=False)
gt.round(4)

**出力の見方**: 低下量が大きい群ほど、**その情報を壊すと予測が悪くなる**＝効いている群です。
個別の特徴量では目立たなかった**水蒸気（露点・湿度）**や**午前の日照・日射**が、群にすると大きく出ることがあります（似た特徴量が互いに代役をするため）。

> ⚠️ **注意**: ここは**評価データが少ない簡易版**です。差は**ぶれの範囲内**かもしれません（ノート03）。このプロジェクトの本番の評価（15年のleave-one-year-out）では、
> 那覇は「午前の日照・日射」「水蒸気」「風」が上位、名護は「風」「時刻・月」が上位でした（REPORT.md）。**地点によって効く群が違う**のも、重要な発見です。

## まとめ
- **impurity** は速いが、**連続値・値の種類が多い特徴量を過大評価**しやすい。
- **permutation** は**評価データでの本当の効き**を測る。値が0付近・マイナスなら、効いていない。
- **似た特徴量は群にまとめて**並べ替える。個別だと、代役がいて低く見える。
- 重要度は**「因果」ではなく「そのモデルが使っているか」**を表す。

## 演習
1. **1.** の `noise_cont` の `rng.normal(size=n)` を `rng.integers(0, 2, n)`（0か1）に変えてください（観察点: impurity の偏りは消えるか。値の種類が少ないと、偶然うまく分けにくい）。
2. **1.** の `min_samples_leaf=1` を `50` に変えてください（観察点: 木を浅くすると、`noise_cont` の impurity は下がるか）。
3. **2.** の `n_repeats=10` を `n_repeats=3` にしてください（観察点: ばらつきの値はどう変わるか。並べ替えの回数が少ないと不安定）。
4. **3.** の `groups` から `"水蒸気（露点・湿度）"` を、`"露点": ["dew"]`, `"湿度": ["humid"]`, `"露点差": ["dewdep"]` の3つに分けてください（観察点: 3つの低下量の合計は、まとめたときの低下量より小さいか）。